# C.3 follow-up — n=10 synthetic confirmation + Path β (WikiText-2)

**Why this notebook.** The 2026-05-27 `colab_c3_lr_pull_sweep` ran 5 cells at n=3. All CI-overlapping. The largest single Δ was Cell D (lr_pull=1.0, n_events=3000, synthetic) at +0.026 in default/tight — borderline; n=10 confirmation needed. WikiText-2 (Path β) was always going to be the next path on the falsification ladder regardless. This notebook does both in parallel.

**Three parallel conditions:**

| Tag | Purpose | corpus | lr_pull | n_events | n_seeds |
|---|---|---|---:|---:|---:|
| `n10_synthetic_cellD` | Confirm Cell D at graduation-level n | synthetic | 1.0 | 3000 | 10 |
| `n10_wikitext_best` | **Path β headline** at best-of-sweep config | wikitext-2 | 1.0 | 3000 | 10 |
| `n3_wikitext_base` | Path β sanity at default config | wikitext-2 | 0.1 | 1000 | 3 |

**Graduation gate.** n=10 with the genuine shuffled-token control is the C.3 graduation gate per [phase-3-deep-dive.md:180-189](https://github.com/Dypatterson/Neuro-AI/blob/main/notes/emergent-codebook/phase-3-deep-dive.md). If `n10_wikitext_best` shows CI-disjoint in any stratum with populated trials in both std and ctrl, **that's a Phase 3 graduation candidate** worth a numbered report.

**Reliability machinery (same as sweep notebook):**
- Cell 1 applies both patches (CLI flags + kernel-trick eigvalsh fix).
- WikiText-2 pre-warmed in the parent process before fork, so subprocesses don't race on the HF download.
- Cell 7 staggers subprocess launches by 1.5 s.
- Cell 7 auto-prints the last 50 lines of any failed subprocess's log.

In [ ]:
# 1. Clone the repo and apply three patches:
#    (a) --lr-pull / --lr-push CLI flags in the C.3 driver.
#    (b) Kernel-trick eigvalsh fix in consolidation.py (CUDA-stable).
#    (c) Switch wikitext loader to "Salesforce/wikitext" namespace
#        (recent huggingface_hub requires namespace/name; bare "wikitext" → HfUriError).
%cd /content
!rm -rf Neuro-AI
!git clone https://github.com/Dypatterson/Neuro-AI.git
%cd Neuro-AI
!git checkout codex/phase5-prime-bundle-first-scene-memory
!git log --oneline -3

import subprocess
pre_pull = subprocess.run(['grep', '-c', '--', '--lr-pull', 'experiments/c3_phase3_exit_criterion.py'], capture_output=True, text=True).stdout.strip()
pre_gram = subprocess.run(['grep', '-c', 'kernel trick', 'src/energy_memory/phase4/consolidation.py'], capture_output=True, text=True).stdout.strip()
pre_wt   = subprocess.run(['grep', '-c', 'Salesforce/wikitext', 'src/energy_memory/phase2/corpus.py'], capture_output=True, text=True).stdout.strip()
print(f'pre-patch tracers: --lr-pull = {pre_pull}; kernel-trick gram = {pre_gram}; Salesforce/wikitext = {pre_wt}')

patch = r'''diff --git a/experiments/c3_phase3_exit_criterion.py b/experiments/c3_phase3_exit_criterion.py
--- a/experiments/c3_phase3_exit_criterion.py
+++ b/experiments/c3_phase3_exit_criterion.py
@@ -576,6 +576,8 @@ def _run_single_seed_condition(
     k: int,
     alpha_anti: float,
     repulsion_step_size: float,
+    lr_pull: float,
+    lr_push: float,
     device: str,
     repo_root: Path,
     wikitext_corpus: Optional[_WikiTextCorpus] = None,
@@ -756,6 +758,8 @@ def _run_single_seed_condition(
             vocab_size=vocab_size,
             n_events=n_consolidation_events,
             device=device,
+            lr_pull=lr_pull,
+            lr_push=lr_push,
             repulsion_step_size=repulsion_step_size,
         )
 
@@ -838,6 +842,8 @@ def run(
     n_consolidation_events: int = 1000,
     alpha_anti: float = 0.0,
     repulsion_step_size: float = 0.0,
+    lr_pull: float = 0.1,
+    lr_push: float = 0.05,
     device: str,
     output_dir: Path,
     repo_root: Path,
@@ -919,6 +925,8 @@ def run(
                     k=k,
                     alpha_anti=alpha_anti,
                     repulsion_step_size=repulsion_step_size,
+                    lr_pull=lr_pull,
+                    lr_push=lr_push,
                     device=device,
                     repo_root=repo_root,
                     wikitext_corpus=wikitext_corpus,
@@ -1010,6 +1018,8 @@ def run(
             "substrate_repulsion_active": bool(
                 alpha_anti > 0.0 and repulsion_step_size > 0.0
             ),
+            "lr_pull": float(lr_pull),
+            "lr_push": float(lr_push),
             "operating_point": {
                 "D": D,
                 "landscape_size": landscape_size,
@@ -1370,6 +1380,27 @@ def main(argv: Optional[Sequence[str]] = None) -> int:
             "smoke (no inter-atom-separability force)."
         ),
     )
+    parser.add_argument(
+        "--lr-pull",
+        type=float,
+        default=0.1,
+        help=(
+            "Per-event consolidation pull learning rate (OnlineCodebookUpdater "
+            "lr_pull). Default 0.1 matches the existing Path α smoke. Sweep "
+            "above this to test whether consolidation strength is too weak "
+            "to express corpus-specific learning at the synthetic operating "
+            "point."
+        ),
+    )
+    parser.add_argument(
+        "--lr-push",
+        type=float,
+        default=0.05,
+        help=(
+            "Per-event consolidation push learning rate (OnlineCodebookUpdater "
+            "lr_push). Default 0.05 matches the existing Path α smoke."
+        ),
+    )
     parser.add_argument(
         "--repulsion-step-size",
         type=float,
@@ -1468,6 +1499,8 @@ def main(argv: Optional[Sequence[str]] = None) -> int:
         n_consolidation_events=args.n_consolidation_events,
         alpha_anti=args.alpha_anti,
         repulsion_step_size=args.repulsion_step_size,
+        lr_pull=args.lr_pull,
+        lr_push=args.lr_push,
         device=args.device,
         output_dir=output_dir,
         repo_root=repo_root,
diff --git a/src/energy_memory/phase4/consolidation.py b/src/energy_memory/phase4/consolidation.py
--- a/src/energy_memory/phase4/consolidation.py
+++ b/src/energy_memory/phase4/consolidation.py
@@ -639,10 +639,29 @@ class ConsolidationState:
         # Hermitian Gram of centered basin members. For complex (FHRR)
         # tensors, diffs.conj().T @ diffs is Hermitian → real eigenvalues
         # via torch.linalg.eigh.
-        sigma = (diffs.conj().transpose(-1, -2) @ diffs) / float(n)
+        # Compute the eigenvalues of σ = diffs.conj().T @ diffs / n via the
+        # n×n Gram matrix gram = diffs @ diffs.conj().T / n instead of the
+        # D×D scatter matrix. The two matrices share exactly the same set
+        # of non-zero eigenvalues (standard "kernel trick" identity); the
+        # D×D form additionally carries (D - n) trivial zero eigenvalues
+        # because rank(σ) ≤ n_members ≤ basin_trace_buffer_size (64) ≪ D
+        # (4096 by default in this project). That (D - n) zero subspace
+        # makes σ numerically ill-conditioned at the precision available
+        # to torch.linalg.eigvalsh — observed on Colab CUDA at 2026-05-27
+        # as LinAlgError 4095 and even on CPU LAPACK as LinAlgError 5/12.
+        # The n×n Gram path is full-rank for non-degenerate samples and
+        # an order of magnitude smaller (4 KB vs 16 MB at D=4096, n=8).
+        # Mathematically byte-identical at the λ_1 / λ_2 layer used below;
+        # the C.2.2 dynamic's behavior is unchanged.
+        gram = (diffs @ diffs.conj().transpose(-1, -2)) / float(n)
         # Eigh returns ascending eigenvalues. Take top two: λ_1 (last),
         # λ_2 (second-to-last). All ops stay on-device.
-        eigvals = torch.linalg.eigvalsh(sigma)
+        try:
+            eigvals = torch.linalg.eigvalsh(gram)
+        except torch._C._LinAlgError:
+            # Defensive: keep the CPU fallback in case some pathological
+            # input still trips cuSOLVER (e.g. identical basin members).
+            eigvals = torch.linalg.eigvalsh(gram.cpu()).to(gram.device)
         lam_1 = eigvals[-1]
         lam_2 = eigvals[-2] if eigvals.shape[0] >= 2 else torch.zeros_like(lam_1)
         # Clamp at 0 — eigh may return tiny negatives for near-singular Σ.
diff --git a/src/energy_memory/phase2/corpus.py b/src/energy_memory/phase2/corpus.py
--- a/src/energy_memory/phase2/corpus.py
+++ b/src/energy_memory/phase2/corpus.py
@@ -113,7 +113,13 @@ def load_repo_sample_splits(repo_root: Path) -> Dict[str, List[str]]:
 def load_wikitext_splits(name: str = "wikitext-2-raw-v1") -> Dict[str, List[str]]:
     if load_dataset is None:  # pragma: no cover - exercised only when dependency missing
         raise ModuleNotFoundError("datasets is required to load WikiText-2")
-    dataset = load_dataset("wikitext", name)
+    # Use the canonical Salesforce/wikitext namespace. The bare "wikitext"
+    # form worked with older HF stacks but recent huggingface_hub versions
+    # (~0.30+) ship a stricter HF URI parser that rejects any repo id
+    # without an explicit namespace, raising HfUriError. The Salesforce
+    # mirror is the current canonical home of the dataset; config names
+    # ("wikitext-2-raw-v1", "wikitext-103-raw-v1", ...) are unchanged.
+    dataset = load_dataset("Salesforce/wikitext", name)
     return {
         "train": [row["text"] for row in dataset["train"]],
         "validation": [row["text"] for row in dataset["validation"]],
'''

with open('/tmp/c3_combined.patch', 'w') as f:
    f.write(patch)
check = subprocess.run(['git', 'apply', '--check', '/tmp/c3_combined.patch'], capture_output=True, text=True)
if check.returncode == 0:
    subprocess.check_call(['git', 'apply', '/tmp/c3_combined.patch'])
    print('combined patch applied.')
else:
    if int(pre_pull or '0') >= 1 and int(pre_gram or '0') >= 1 and int(pre_wt or '0') >= 1:
        print('all three patches already in branch — skipping apply.')
    else:
        print('PATCH APPLY FAILED:'); print(check.stderr)
        raise SystemExit('Cannot continue.')

post_pull = subprocess.check_output(['grep', '-c', '--', '--lr-pull', 'experiments/c3_phase3_exit_criterion.py']).decode().strip()
post_gram = subprocess.check_output(['grep', '-c', 'kernel trick', 'src/energy_memory/phase4/consolidation.py']).decode().strip()
post_wt   = subprocess.check_output(['grep', '-c', 'Salesforce/wikitext', 'src/energy_memory/phase2/corpus.py']).decode().strip()
print(f'post-patch tracers: --lr-pull = {post_pull}; kernel-trick = {post_gram}; Salesforce/wikitext = {post_wt}')
assert int(post_pull) >= 1 and int(post_gram) >= 1 and int(post_wt) >= 1, 'patches missing'

In [ ]:
# 2. Mount Drive.
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/neuro-ai/results', exist_ok=True)
print('Drive mounted.')

In [ ]:
# 3. Install deps. WikiText needs the `datasets` library.
#    Pin datasets<3 because the loader at src/energy_memory/phase2/corpus.py:116
#    calls `load_dataset("wikitext", name)` — the plain dataset name. From
#    datasets >=3.0 the HF URI parser requires `namespace/name` (e.g.
#    `Salesforce/wikitext`) and rejects the bare `wikitext` form with
#    HfUriError. Pinning here is the minimum-blast-radius fix; the loader
#    itself can be updated in a follow-up.
!pip install -q "datasets<3"
import sys, torch, numpy as np, datasets
print(f'python: {sys.version.split()[0]} | torch: {torch.__version__} | numpy: {np.__version__} | datasets: {datasets.__version__}')
print(f'cuda available: {torch.cuda.is_available()}; device 0: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')

In [ ]:
# 4. Pre-warm the WikiText-2 cache so 2 subprocesses don't race on the HuggingFace download.
#    Run from the parent (CPU only — does not init CUDA).
import sys; sys.path.insert(0, '/content/Neuro-AI/src')
from energy_memory.phase2.corpus import load_corpus_splits
from pathlib import Path
print('warming wikitext-2-raw-v1 cache (parent process, CPU only)...')
splits = load_corpus_splits('wikitext', Path('/content/Neuro-AI'), wikitext_name='wikitext-2-raw-v1')
print(f'  train: {len(splits["train"])} rows')
print(f'  validation: {len(splits["validation"])} rows')
print(f'  test: {len(splits["test"])} rows')
print('cache warmed; subprocesses will reuse it.')
del splits
import gc; gc.collect()

In [ ]:
# 5. GPU info.
!nvidia-smi --query-gpu=name,memory.total,compute_mode --format=csv
!nvidia-smi --query-compute-apps=pid,process_name,used_memory --format=csv

In [ ]:
# 6. SMOKE TEST — one tiny synthetic subprocess to confirm runtime is sane.
import subprocess, sys, os
from pathlib import Path
os.environ['PYTHONPATH'] = '/content/Neuro-AI/src'

smoke_out = Path('reports/c3_followup_smoke_2026-05-27')
smoke_out.mkdir(parents=True, exist_ok=True)
smoke_log = Path('reports/c3_followup_smoke.log')
cmd = [sys.executable, 'experiments/c3_phase3_exit_criterion.py',
       '--seeds', '0', '--device', 'cuda',
       '--lr-pull', '1.0', '--lr-push', '0.05',
       '--n-consolidation-events', '100',
       '--alpha-anti', '0.01', '--repulsion-step-size', '0.05',
       '--output-dir', str(smoke_out)]
print('cmd:', ' '.join(cmd))
with smoke_log.open('w') as logf:
    rc = subprocess.call(cmd, stdout=logf, stderr=subprocess.STDOUT)
print(f'smoke exit code: {rc}')
print(f'json written: {(smoke_out / "c3_summary.json").exists()}')
print()
print('=== smoke log (last 60 lines) ===')
!tail -60 {smoke_log}
if rc != 0:
    raise SystemExit('Smoke failed — abort.')
print('\nSmoke OK.')

In [ ]:
# 7. PARALLEL launch — 3 conditions:
#    1. n=10 synthetic Cell D confirmation (lr_pull=1.0, n_events=3000)
#    2. n=10 wikitext at best-of-sweep (lr_pull=1.0, n_events=3000) — Path β headline
#    3. n=3  wikitext at default      (lr_pull=0.1, n_events=1000) — Path β sanity
import subprocess, os, time, signal, sys
from pathlib import Path
os.environ['PYTHONPATH'] = '/content/Neuro-AI/src'
PY = sys.executable

SEEDS_N10 = '0,1,2,3,4,5,6,7,8,9'
SEEDS_N3  = '0,1,2'

CELLS = [
    # (tag, lr_pull, n_events, seeds, corpus_source)
    ('n10_synthetic_cellD', 1.0, 3000, SEEDS_N10, 'synthetic'),
    ('n10_wikitext_best',   1.0, 3000, SEEDS_N10, 'wikitext'),
    ('n3_wikitext_base',    0.1, 1000, SEEDS_N3,  'wikitext'),
]

log_root = Path('reports/c3_followup_logs')
log_root.mkdir(parents=True, exist_ok=True)

def launch(tag, lr_pull, n_events, seeds, corpus_source):
    out_dir = f'reports/c3_followup_{tag}_2026-05-27'
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    log_path = log_root / f'{tag}.log'
    logf = open(log_path, 'w')
    cmd = [PY, 'experiments/c3_phase3_exit_criterion.py',
           '--seeds', seeds, '--device', 'cuda',
           '--lr-pull', str(lr_pull), '--lr-push', '0.05',
           '--n-consolidation-events', str(n_events),
           '--alpha-anti', '0.01', '--repulsion-step-size', '0.05',
           '--corpus-source', corpus_source,
           '--output-dir', out_dir]
    proc = subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT)
    return proc, logf, out_dir, log_path

def snapshot(remaining, t0):
    elapsed = (time.time() - t0) / 60
    print(f'  --- snapshot at {elapsed:.1f} min ---')
    try:
        gpu = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=memory.used,utilization.gpu', '--format=csv,noheader'],
            stderr=subprocess.DEVNULL).decode().strip()
        print(f'  GPU: {gpu}')
    except Exception as e:
        print(f'  GPU snapshot failed: {e}')
    for tag, (proc, logf, out_dir, log_path) in remaining.items():
        size = log_path.stat().st_size if log_path.exists() else 0
        try:
            line = subprocess.check_output(['tail', '-1', str(log_path)],
                stderr=subprocess.DEVNULL).decode().strip()[:90]
        except Exception:
            line = ''
        print(f'  {tag:>22}: {size:>7}b  {line}')

def kill_all(remaining):
    for tag, (proc, logf, out_dir, log_path) in remaining.items():
        try:
            proc.send_signal(signal.SIGKILL); logf.close()
        except Exception:
            pass

procs = {}
for spec in CELLS:
    procs[spec[0]] = launch(*spec)
    time.sleep(1.5)
print(f'launched {len(procs)} cells  pids={[p[0].pid for p in procs.values()]}  ({time.strftime("%H:%M:%S")})')

t0 = time.time()
remaining = dict(procs)
poll_count = 0
try:
    while remaining:
        done_this_round = []
        for tag, (proc, logf, out_dir, log_path) in remaining.items():
            rc = proc.poll()
            if rc is not None:
                logf.close()
                elapsed = (time.time() - t0) / 60
                ok = 'OK' if rc == 0 else f'FAILED (exit={rc})'
                json_exists = Path(out_dir, 'c3_summary.json').exists()
                print(f'  [{elapsed:5.1f} min] {tag:>22}: {ok}  json={json_exists}')
                if rc != 0:
                    print(f'    --- last 50 lines of {log_path} ---')
                    try:
                        out = subprocess.check_output(['tail', '-50', str(log_path)],
                            stderr=subprocess.DEVNULL).decode()
                        for line in out.splitlines():
                            print(f'    | {line}')
                    except Exception as e:
                        print(f'    | (could not read log: {e})')
                    print(f'    --- end log ---')
                done_this_round.append(tag)
        for tag in done_this_round:
            del remaining[tag]
        if remaining:
            poll_count += 1
            if poll_count % 3 == 0:
                snapshot(remaining, t0)
            time.sleep(30)
except KeyboardInterrupt:
    print('\n!!! Interrupted !!!')
    kill_all(remaining); raise

print(f'\nALL DONE in {(time.time()-t0)/60:.1f} min')
!nvidia-smi --query-gpu=memory.used,memory.total,utilization.gpu --format=csv

In [ ]:
# 7b. EMERGENCY kill.
import subprocess, signal, os
killed = 0
for line in subprocess.check_output(['ps', '-eo', 'pid,cmd']).decode().splitlines():
    if 'c3_phase3_exit_criterion' in line and 'grep' not in line:
        try:
            pid = int(line.split()[0])
            os.kill(pid, signal.SIGKILL); print(f'  killed {pid}'); killed += 1
        except Exception as e:
            print(f'  err: {e}')
print(f'killed {killed} workers')

In [ ]:
# 8. Copy results + logs to Drive.
import shutil, os
dst_root = '/content/drive/MyDrive/neuro-ai/results/c3_followup_2026-05-27'
os.makedirs(dst_root, exist_ok=True)
TAGS = ['n10_synthetic_cellD', 'n10_wikitext_best', 'n3_wikitext_base']
for tag in TAGS:
    src = f'reports/c3_followup_{tag}_2026-05-27'
    if os.path.isdir(src):
        shutil.copytree(src, f'{dst_root}/c3_followup_{tag}_2026-05-27', dirs_exist_ok=True)
if os.path.isdir('reports/c3_followup_logs'):
    shutil.copytree('reports/c3_followup_logs', f'{dst_root}/colab_logs', dirs_exist_ok=True)
print('results in', dst_root)
!ls {dst_root}

In [ ]:
# 9. In-notebook headline table.
import json
from pathlib import Path

CELLS = [
    # (tag, lr_pull, n_events, corpus, expected_n_seeds)
    ('n10_synthetic_cellD', 1.0, 3000, 'synthetic', 10),
    ('n10_wikitext_best',   1.0, 3000, 'wikitext',  10),
    ('n3_wikitext_base',    0.1, 1000, 'wikitext',  3),
]
STRATA = ('tight', 'spread', 'borderline')

print(f'{"cell":>22} {"lrP":>5} {"n_ev":>5} {"corpus":>10} {"mode":>11} {"stratum":>11}  '
      f'{"std R@K":>22}  {"ctrl R@K":>22}  {"Δ":>7}  disjoint?  n_std  n_ctrl')
for tag, lr_pull, n_events, corpus, n_seeds in CELLS:
    p = Path(f'reports/c3_followup_{tag}_2026-05-27/c3_summary.json')
    if not p.exists():
        print(f'{tag:>22} {lr_pull:>5} {n_events:>5} {corpus:>10}  MISSING')
        continue
    d = json.loads(p.read_text())
    agg = d['aggregated']
    for mode in d['header']['theta_prime_modes_run']:
        for stratum in STRATA:
            s = agg[mode]['standard'][stratum]
            c = agg[mode]['shuffled_control'][stratum]
            dl = agg[mode]['delta_standard_minus_control'][stratum]
            if s['trials'] == 0 and c['trials'] == 0:
                continue
            std_str  = f'{s["recall_at_k"]:.3f} [{s["wilson_lower"]:.3f},{s["wilson_upper"]:.3f}]'
            ctrl_str = f'{c["recall_at_k"]:.3f} [{c["wilson_lower"]:.3f},{c["wilson_upper"]:.3f}]'
            disj = 'YES' if dl['ci_disjoint_standard_beats_control'] else 'no'
            print(f'{tag:>22} {lr_pull:>5} {n_events:>5} {corpus:>10} {mode:>11} {stratum:>11}  '
                  f'{std_str:>22}  {ctrl_str:>22}  {dl["delta_recall_at_k"]:>+7.3f}  {disj:>9}  '
                  f'{s["trials"]:>5}  {c["trials"]:>5}')
print()
print('Interpretation:')
print('  n10_wikitext_best CI-disjoint in any stratum  →  Phase 3 graduation candidate (worth a numbered report).')
print('  n10_synthetic_cellD CI-disjoint               →  confirms Cell D was real, escalate to n=10 wikitext.')
print('  both null                                     →  Path γ (Phase 3 mechanism redesign) is the remaining honest option.')